# Phase 6b Stage 2 — train the two ablation finalists

Trains `w125` (P3/P4/P5, width 0.125) and `p34` (P3/P4, width 0.25) from random init and compares accuracy — the piece Phase 6a deliberately left undone (it measured flash/RAM cost only, no training).

**Moved here from a local M4 Pro run** that hung the machine: `--cache ram` decodes this dataset to ~19GB, and on Apple Silicon's unified memory that competes directly with the GPU's own pool. Colab's GPU has dedicated VRAM separate from system RAM, so that failure mode doesn't apply here — but free-tier Colab system RAM (~12GB) is still *smaller* than the Mac's 24GB, so this notebook keeps `--cache` off regardless. Plain disk reads are fast enough on a real GPU.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine). Then run cells top to bottom.

**You'll need:** a Kaggle account + API token (kaggle.com -> Settings -> Create New Token -> downloads `kaggle.json`). Cell 3 will prompt you to upload it.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone -b phase/six https://github.com/akash-reddy-k/edgeAi.git
%cd edgeAi
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt kaggle

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime > GPU before continuing")

## 2. Kaggle credentials + dataset download

Regenerating the dataset from source (rather than uploading your local 874MB copy) keeps this reproducible from a clean clone, same as the README's local instructions. `prepare_dataset.py` is deterministic — fixed val-scene split, fixed pseudo-label confidence threshold — so this reproduces the same train/val split and labels your local run used.

In [ ]:
from google.colab import files
print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
uploaded = files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nikanvasei/shanghaitech-campus-dataset-test -p data/SHANGHAI_Test --unzip

In [ ]:
# Kaggle's --unzip sometimes nests one level deep depending on how the
# uploader zipped it. Find frames/ and SHANGHAI_test.txt wherever they
# landed under data/ and normalise to the layout prepare_dataset.py expects.
import shutil
from pathlib import Path

base = Path("data/SHANGHAI_Test")
root = Path("data")

if not (base / "frames").exists():
    candidates = [p for p in root.rglob("frames") if p.is_dir()]
    assert candidates, "No frames/ directory found under data/ — inspect the download manually"
    shutil.move(str(candidates[0]), str(base / "frames"))

if not (base / "SHANGHAI_test.txt").exists():
    candidates = list(root.rglob("SHANGHAI_test.txt"))
    assert candidates, "No SHANGHAI_test.txt found under data/ — inspect the download manually"
    shutil.move(str(candidates[0]), str(base / "SHANGHAI_test.txt"))

n_scenes = len(list((base / "frames").iterdir()))
print(f"OK — {n_scenes} scenes under {base}/frames")

## 3. Base weights + pseudo-labeled dataset

`prepare_dataset.py` uses stock `yolov8n.pt` as a pseudo-labeler (COCO -> campus domain adaptation), same as Phase 3 locally.

In [ ]:
import os, shutil
from ultralytics import YOLO

os.makedirs("models", exist_ok=True)
YOLO("yolov8n.pt")  # downloads to cwd if not already cached
if os.path.exists("yolov8n.pt") and not os.path.exists("models/yolov8n.pt"):
    shutil.move("yolov8n.pt", "models/yolov8n.pt")
print("models/yolov8n.pt present:", os.path.exists("models/yolov8n.pt"))

In [ ]:
!python prepare_dataset.py

## 4. Train the two Stage 2 finalists

Both from random init (`--variant` implies `pretrained=False` in `train.py` — see its docstring for why). `--device 0` targets the Colab GPU. No `--cache`: kept off deliberately, see the caveat at the top.

In [ ]:
!python train.py --variant w125 --epochs 40 --patience 15 --batch 16 --device 0

In [ ]:
!python train.py --variant p34 --epochs 40 --patience 15 --batch 16 --device 0

## 5. Package results for download

Zips both run directories (`results.csv`, `args.yaml`, plots) and the best weights. Unzip locally into the same paths — `training_runs/*` -> `runs/detect/results/training/`, `models/*.pt` -> `models/` — and everything downstream (the Stage 2 comparison, README) reads it exactly like a local run.

In [ ]:
import shutil, os
from google.colab import files

stage_dir = "stage2_package"
os.makedirs(f"{stage_dir}/models", exist_ok=True)
shutil.copytree("runs/detect/results/training", f"{stage_dir}/training_runs", dirs_exist_ok=True)

for f in ["models/yolov8_ablation_w125.pt", "models/yolov8_ablation_p34.pt"]:
    if os.path.exists(f):
        shutil.copy(f, f"{stage_dir}/models/")
    else:
        print(f"WARNING: {f} missing — training may not have completed")

shutil.make_archive("stage2_results", "zip", stage_dir)
files.download("stage2_results.zip")

In [ ]:
# Quick summary printed here too, in case the download step is skipped —
# paste this back into the conversation if you'd rather not transfer files.
import pandas as pd

for variant in ["w125", "p34"]:
    csv_path = f"runs/detect/results/training/yolov8_ablation_{variant}/results.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        print(f"--- {variant} ---")
        print(f"  epochs run : {len(df)}")
        print(f"  mAP50      : {last['metrics/mAP50(B)']:.4f}")
        print(f"  mAP50-95   : {last['metrics/mAP50-95(B)']:.4f}")
        print(f"  Precision  : {last['metrics/precision(B)']:.4f}")
        print(f"  Recall     : {last['metrics/recall(B)']:.4f}\n")